El Objetivo: Entrenar un modelo de regresión (ej. RandomForestRegressor) donde la variable objetivo sea ltv_revenue [conversación previa].
Segmentación "Shark": Utiliza los hallazgos de tu Data Forensics. El modelo debe pesar fuertemente si el lead pertenece a watches o health_beauty, ya que detectaste que un solo lead de relojería en canal unknown puede valer $113,629
.
Justificación Senior: Vende el concepto de Expected Value (Valor Esperado). Un Senior le dice al negocio: "No llames al lead con 90% de probabilidad de cerrar $100; llama al lead con 20% de probabilidad de cerrar $113,000"
.

In [3]:
import pandas as pd
import joblib
import pyarrow.parquet as pq
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [4]:
# 1. Preparación de Features (Mentalidad de Producto)
print("⏳ Cargando backup local en formato Parquet...")
table = pq.read_table('C:\\Users\\User\\Desktop\\Software y Clases\\BigData\\OList\\olist-project-sa\\Data\\Processed\\abt_marketing.parquet')

# 2. ELIMINAR los metadatos de Pandas que causan el conflicto
# Esto quita la etiqueta 'dbdate' que confunde a numpy
table_sin_metadata = table.replace_schema_metadata(None)

# 3. Convertir a Pandas (ahora sí debería funcionar)
df_rf = table_sin_metadata.to_pandas()
print(f"✅ Datos cargados con éxito: {df_rf.shape}")

⏳ Cargando backup local en formato Parquet...
✅ Datos cargados con éxito: (8000, 11)


In [6]:
# Senior Hint: Entrenamos el regresor SOLO con los leads que SÍ convirtieron (vendedores reales)
# para que el modelo aprenda qué características definen un negocio de $100k vs uno de $1k.
df_reg = df_rf[df_rf['converted'] == 1].copy()

In [7]:
# Feature Engineering inyectando tus hallazgos de Data Forensics
high_value_list = ['watches', 'health_beauty', 'audio_video_electronics']
df_reg['is_high_value_segment'] = df_reg['business_segment'].isin(high_value_list).astype(int)

features = ['origin', 'lead_type', 'is_high_value_segment']
X = pd.get_dummies(df_reg[features], drop_first=True)
y = df_reg['ltv_revenue'] # Nuestra variable objetivo económica

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Entrenando Regresor de Valor con {X_train.shape} vendedores reales...")

Entrenando Regresor de Valor con (673, 18) vendedores reales...


In [10]:
# Usamos Random Forest Regressor por su robustez ante outliers como el lead de $113k
regressor = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
regressor.fit(X_train, y_train)

# Evaluación
y_pred = regressor.predict(X_test)
print(f"📊 MAE (Error Medio Absoluto): ${mean_absolute_error(y_test, y_pred):.2f}")

# Guardar activos para Streamlit
joblib.dump(regressor, 'C:\\Users\\User\\Desktop\\Software y Clases\\BigData\\OList\\olist-project-sa\\Model\\ltv_regressor_model.joblib')
joblib.dump(X_train.columns.tolist(), 'C:\\Users\\User\\Desktop\\Software y Clases\\BigData\\OList\\olist-project-sa\\Model\\regressor_features.joblib')
print("✅ Regresor de Ingresos guardado con éxito.")

📊 MAE (Error Medio Absoluto): $1966.45
✅ Regresor de Ingresos guardado con éxito.
